# V80 - Recency-weighted chat pooling

Pre-registered plan: every text view concatenates all chats equally, but intent drifts and the newest message may carry the current need. Message-level TF-IDF pooled by linear recency weights faces mean pooling under identical folds and seed. Timestamps order messages within user. Same locks.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 9191
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v80")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Mean vs recency pooling, same folds

Messages sort by timestamp per user. Recency weights run 1 to n normalized. Single-chat and chatless users are unaffected by construction.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv", parse_dates=["timestamp"])
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
chats = chats.sort_values(["user_id", "timestamp"])
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3)
M = vec.fit_transform(chats["user_chat_text"].fillna("").tolist())
row_of = {}
for i, (u, t) in enumerate(zip(chats["user_id"], chats["user_chat_text"])):
    row_of.setdefault(u, []).append(i)
F_mean = np.zeros((len(all_users), M.shape[1]))
F_rec = np.zeros((len(all_users), M.shape[1]))
for i, u in enumerate(all_users):
    rows = row_of.get(u, [])
    if rows:
        V = M[rows].toarray()
        F_mean[i] = V.mean(0)
        w = np.arange(1, len(rows) + 1, dtype=float)
        F_rec[i] = (V * (w / w.sum())[:, None]).sum(0)
print(f"users with >1 chat: {sum(len(v) > 1 for v in row_of.values())}")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_m, fid = ridge_oof(F_mean[:n_train])
oof_r, _ = ridge_oof(F_rec[:n_train])
base = ndcg5(Y, oof_m)
d = ndcg5(Y, oof_r) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_r[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"mean-pool={base.mean():.5f} recency={ndcg5(Y, oof_r).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT recency pooling" if (lo > GATE_FLOOR and wf >= 0) else "KEEP equal pooling; recency adds nothing")